# Definitive CAMA/RAMA Meta-Analysis: 15 Experiments Across Iterations 2-5

This notebook demonstrates a comprehensive meta-analysis evaluating CAMA (Contextual Attention-weighted Moment Aggregation) and RAMA (Rank-Aware Moment Aggregation) across 15 experiments. It implements a 6-phase analysis pipeline:

- **Phase A**: GRADE-adapted evidence grading of experiments
- **Phase B**: Per-task best-evidence selection across 8 tasks
- **Phase C**: DerSimonian-Laird random-effects meta-analysis under 4 scenarios
- **Phase D**: Sum-comparison sub-analysis
- **Phase E**: Critical diagnostics (gate stasis, degenerate baselines, epoch sensitivity)
- **Phase F**: Publication readiness assessment

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — all imports are from pre-installed packages

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
from typing import Any

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-9b0386-rank-aware-moment-aggregation-diversity-/main/evaluation_iter6_definitive_cama/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
all_effects = data["all_effects"]
sum_comparison_data = data["sum_comparison_data"]
diagnostics_data = data["diagnostics_data"]

# Convert None d values to NaN for consistency
for eff in all_effects:
    if eff["d"] is None:
        eff["d"] = float("nan")
    if eff.get("ci_lower") is None:
        eff["ci_lower"] = float("nan")
    if eff.get("ci_upper") is None:
        eff["ci_upper"] = float("nan")
    if eff.get("p_value") is None:
        eff["p_value"] = float("nan")

print(f"Loaded {len(all_effects)} effect sizes from {len(set(e['exp_id'] for e in all_effects))} experiments")
print(f"Tasks covered: {sorted(set(e['task'] for e in all_effects))}")

## Configuration

Analysis parameters controlling the meta-analysis pipeline.

In [ ]:
# ---- Tunable configuration ----
ALPHA = 0.05                    # Significance level for CIs and p-values
GATE_STASIS_LO = 0.499          # Lower bound for gate stasis detection
GATE_STASIS_HI = 0.501          # Upper bound for gate stasis detection
INFLATED_D_THRESHOLD = 10.0     # Cohen's d above this is flagged as inflated
DEGENERATE_CV_THRESHOLD = 0.001 # Coefficient of variation threshold for degenerate baselines
MIN_SEEDS_FOR_TTEST = 2         # Minimum seeds required for paired t-test

## Statistical Helper Functions

Core statistical functions for computing effect sizes and running the DerSimonian-Laird random-effects meta-analysis.

In [ ]:
def safe_float(v: Any, default: float = float("nan")) -> float:
    """Convert to float, returning default for None/NaN/non-numeric."""
    if v is None:
        return default
    try:
        f = float(v)
        return f if math.isfinite(f) else default
    except (TypeError, ValueError):
        return default


def cohens_d(baseline: list[float], method: list[float],
             lower_is_better: bool = False) -> float:
    """Compute Cohen's d with convention: positive = method better."""
    m1, m2 = np.mean(baseline), np.mean(method)
    n1, n2 = len(baseline), len(method)
    s1, s2 = np.std(baseline, ddof=1), np.std(method, ddof=1)
    sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    if sp < 1e-15:
        return 0.0
    if lower_is_better:
        return float((m1 - m2) / sp)
    return float((m2 - m1) / sp)


def cohens_d_ci(d: float, n1: int, n2: int, alpha: float = ALPHA) -> tuple[float, float]:
    """Approximate CI for Cohen's d using non-central t approximation."""
    se = np.sqrt((n1 + n2) / (n1 * n2) + d**2 / (2 * (n1 + n2)))
    z = stats.norm.ppf(1 - alpha / 2)
    return (float(d - z * se), float(d + z * se))


def paired_ttest_p(group1: list[float], group2: list[float]) -> float:
    """Two-sided paired t-test p-value."""
    if len(group1) < MIN_SEEDS_FOR_TTEST or len(group2) < MIN_SEEDS_FOR_TTEST or len(group1) != len(group2):
        return float("nan")
    diffs = [b - a for a, b in zip(group1, group2)]
    if np.std(diffs, ddof=1) < 1e-15:
        return 1.0 if abs(np.mean(diffs)) < 1e-15 else 0.0
    t_stat, p_val = stats.ttest_rel(group2, group1)
    return float(p_val)


def variance_of_d(d: float, n1: int, n2: int) -> float:
    """Approximate sampling variance of Cohen's d."""
    return (n1 + n2) / (n1 * n2) + d**2 / (2 * (n1 + n2))


def dersimonian_laird(effects: list[float], variances: list[float]) -> dict:
    """DerSimonian-Laird random-effects meta-analysis.

    Returns pooled effect, SE, CI, p-value, I-squared, Q, tau-squared.
    """
    k = len(effects)
    if k == 0:
        return {"pooled_d": float("nan"), "pooled_se": float("nan"),
                "pooled_ci_lower": float("nan"), "pooled_ci_upper": float("nan"),
                "pooled_p_value": float("nan"), "I_squared": float("nan"),
                "Q_statistic": float("nan"), "Q_p_value": float("nan"),
                "tau_squared": float("nan"), "n_effects": 0, "sign_consistency": float("nan")}

    d = np.array(effects, dtype=float)
    v = np.array(variances, dtype=float)
    v = np.maximum(v, 1e-10)
    w = 1.0 / v

    # Fixed-effects pooled estimate
    d_fe = float(np.sum(w * d) / np.sum(w))

    # Cochran's Q
    Q = float(np.sum(w * (d - d_fe)**2))
    Q_p = float(1.0 - stats.chi2.cdf(Q, max(k - 1, 1))) if k > 1 else 1.0

    # tau-squared (between-study variance)
    C = float(np.sum(w) - np.sum(w**2) / np.sum(w))
    tau2 = max(0.0, (Q - (k - 1)) / C) if C > 0 and k > 1 else 0.0

    # Random-effects weights
    w_re = 1.0 / (v + tau2)
    d_re = float(np.sum(w_re * d) / np.sum(w_re))
    se_re = float(np.sqrt(1.0 / np.sum(w_re)))

    # I-squared
    I2 = max(0.0, (Q - (k - 1)) / Q * 100) if Q > 0 and k > 1 else 0.0

    # Pooled p-value
    z = d_re / se_re if se_re > 0 else 0.0
    p_val = float(2 * (1 - stats.norm.cdf(abs(z))))

    # Sign consistency
    if d_re >= 0:
        sign_cons = float(np.mean(d >= 0))
    else:
        sign_cons = float(np.mean(d < 0))

    return {
        "pooled_d": round(d_re, 6),
        "pooled_se": round(se_re, 6),
        "pooled_ci_lower": round(d_re - 1.96 * se_re, 6),
        "pooled_ci_upper": round(d_re + 1.96 * se_re, 6),
        "pooled_p_value": round(p_val, 8),
        "I_squared": round(I2, 2),
        "Q_statistic": round(Q, 4),
        "Q_p_value": round(Q_p, 6),
        "tau_squared": round(tau2, 6),
        "n_effects": k,
        "sign_consistency": round(sign_cons, 4),
    }

print("Statistical helper functions defined.")

## Phase A: Evidence Registry

Grade each experiment using a GRADE-adapted framework. Grades range from GREEN (high quality) through YELLOW (moderate) to RED (low quality), with MECHANISM-ONLY for non-comparative studies.

In [ ]:
def phase_a_evidence_registry(all_effects: list[dict]) -> dict:
    """Grade each experiment and return registry."""
    registry = {}
    for eff in all_effects:
        exp_id = eff["exp_id"]
        if exp_id not in registry:
            registry[exp_id] = {
                "grade": eff["grade"],
                "flags": eff["grade_flags"],
                "tasks": [],
            }
        registry[exp_id]["tasks"].append(eff["task"])
        # Upgrade to worst grade
        grade_order = {"MECHANISM-ONLY": 0, "GREEN": 1, "YELLOW": 2, "RED": 3}
        curr = grade_order.get(registry[exp_id]["grade"], 1)
        new = grade_order.get(eff["grade"], 1)
        if new > curr:
            registry[exp_id]["grade"] = eff["grade"]
            registry[exp_id]["flags"] = eff["grade_flags"]
    return registry

registry = phase_a_evidence_registry(all_effects)
print(f"Phase A: {len(registry)} experiments graded\n")
for eid, info in sorted(registry.items()):
    print(f"  {eid:15s}: {info['grade']:15s} | tasks={info['tasks']}")

## Phase B: Per-Task Best Evidence Selection

For each of the 8 benchmark tasks, select the highest-quality experiment (by grade, then seeds, then epochs) and check for sign discrepancy across experiments.

In [ ]:
def phase_b_best_evidence(all_effects: list[dict], registry: dict) -> dict:
    """Select best-evidence experiment per task."""
    task_effects: dict[str, list[dict]] = {}
    for eff in all_effects:
        if eff["grade"] in ["MECHANISM-ONLY"]:
            continue
        # Only include CAMA/RAMA vs mean comparisons (not vs sum)
        if eff.get("baseline_name") == "sum":
            continue
        task = eff["task"]
        if task not in task_effects:
            task_effects[task] = []
        task_effects[task].append(eff)

    best_per_task = {}
    for task, effects in sorted(task_effects.items()):
        grade_priority = {"GREEN": 0, "YELLOW": 1, "RED": 2}
        effects.sort(key=lambda e: (
            grade_priority.get(e["grade"], 3),
            -e["n_seeds"],
            -e.get("epochs", 0),
        ))
        best = effects[0]
        d_values = [e["d"] for e in effects if math.isfinite(e["d"])]
        signs = [1 if d >= 0 else -1 for d in d_values]
        discrepancy = len(set(signs)) > 1 if signs else False

        best_per_task[task] = {
            "best_experiment_id": best["exp_id"],
            "best_d": best["d"],
            "best_ci_lower": best.get("ci_lower", float("nan")),
            "best_ci_upper": best.get("ci_upper", float("nan")),
            "best_p_value": best.get("p_value", float("nan")),
            "n_experiments": len(effects),
            "discrepancy_flag": discrepancy,
            "all_d_values": {e["exp_id"]: round(e["d"], 4) if math.isfinite(e["d"]) else None for e in effects},
            "task_type": best["task_type"],
            "metric": best["metric"],
            "direction": best["direction"],
            "grade": best["grade"],
        }
    return best_per_task

best_per_task = phase_b_best_evidence(all_effects, registry)
print(f"Phase B: {len(best_per_task)} unique tasks with best evidence\n")
for task, info in sorted(best_per_task.items()):
    disc = " *** DISCREPANCY" if info["discrepancy_flag"] else ""
    print(f"  {task:30s}: best={info['best_experiment_id']:12s} d={info['best_d']:8.4f}  "
          f"n_exp={info['n_experiments']}  grade={info['grade']}{disc}")

## Phase C: DerSimonian-Laird Random-Effects Meta-Analysis

Run the meta-analysis under 4 scenarios to assess the overall evidence:
1. GREEN-only experiments (highest quality)
2. GREEN + YELLOW experiments (broader evidence base)
3. Classification tasks only
4. Regression tasks only

In [ ]:
def phase_c_meta_analysis(best_per_task: dict, all_effects: list[dict]) -> dict:
    """Run DL meta-analysis under 4 scenarios."""
    scenarios = {}

    def get_effects_for_scenario(filter_fn):
        effects_list, variances_list = [], []
        for task, info in best_per_task.items():
            if not filter_fn(info):
                continue
            d = info["best_d"]
            if not math.isfinite(d):
                continue
            matching = [e for e in all_effects
                        if e["exp_id"] == info["best_experiment_id"] and e["task"] == task]
            n = matching[0]["n_seeds"] if matching else 5
            v = variance_of_d(d, n, n)
            effects_list.append(d)
            variances_list.append(v)
        return effects_list, variances_list

    # Scenario 1: GREEN only
    effs, vars_ = get_effects_for_scenario(lambda i: i["grade"] == "GREEN")
    scenarios["scenario_1_green_only"] = dersimonian_laird(effs, vars_)

    # Scenario 2: GREEN + YELLOW
    effs, vars_ = get_effects_for_scenario(lambda i: i["grade"] in ["GREEN", "YELLOW"])
    scenarios["scenario_2_green_yellow"] = dersimonian_laird(effs, vars_)

    # Scenario 3: Classification only, GREEN+YELLOW
    effs, vars_ = get_effects_for_scenario(
        lambda i: i["grade"] in ["GREEN", "YELLOW"] and i["task_type"] == "classification"
    )
    scenarios["scenario_3_classification"] = dersimonian_laird(effs, vars_)

    # Scenario 4: Regression only, GREEN+YELLOW
    effs, vars_ = get_effects_for_scenario(
        lambda i: i["grade"] in ["GREEN", "YELLOW"] and i["task_type"] == "regression"
    )
    scenarios["scenario_4_regression"] = dersimonian_laird(effs, vars_)

    return scenarios

scenarios = phase_c_meta_analysis(best_per_task, all_effects)
print("Phase C: Meta-Analysis Results\n")
print(f"{'Scenario':<30s} {'Pooled d':>10s} {'SE':>8s} {'p-value':>10s} {'I2 (%)':>8s} {'n':>4s}")
print("-" * 75)
for sc_name, sc in scenarios.items():
    print(f"{sc_name:<30s} {sc['pooled_d']:10.4f} {sc['pooled_se']:8.4f} "
          f"{sc['pooled_p_value']:10.6f} {sc['I_squared']:8.1f} {sc['n_effects']:4d}")

## Phase D: Sum Comparison Sub-Analysis

Analyze whether simple sum aggregation makes CAMA redundant. If sum performs as well as CAMA, the learned gating mechanism provides no additional value.

In [ ]:
def phase_d_sum_comparison(sum_comparison_data: dict) -> dict:
    """Analyze whether sum aggregation makes CAMA redundant (using pre-computed data)."""
    results = sum_comparison_data

    cama_vs_sum_ds = [v["cama_vs_sum_d"] for v in results.values()
                      if isinstance(v.get("cama_vs_sum_d"), (int, float)) and math.isfinite(v["cama_vs_sum_d"])]
    just_use_sum = all(d <= 0 for d in cama_vs_sum_ds) if cama_vs_sum_ds else False

    # Gap closure analysis
    gap_pcts = []
    for task, v in results.items():
        cama_d = v.get("cama_vs_mean_d", float("nan"))
        sum_d = v.get("sum_vs_mean_d", float("nan"))
        if isinstance(cama_d, (int, float)) and isinstance(sum_d, (int, float)):
            if math.isfinite(cama_d) and math.isfinite(sum_d) and abs(sum_d) > 0.01:
                gap_pct = (cama_d / sum_d) * 100 if sum_d != 0 else float("nan")
                gap_pcts.append(gap_pct)

    summary = {
        "per_task": results,
        "cama_vs_sum_all_d": cama_vs_sum_ds,
        "just_use_sum_verdict": just_use_sum,
        "mean_cama_closes_gap_pct": float(np.mean(gap_pcts)) if gap_pcts else float("nan"),
        "n_tasks_with_sum_comparison": len(cama_vs_sum_ds),
    }
    return summary

sum_analysis = phase_d_sum_comparison(sum_comparison_data)
print("Phase D: Sum Comparison\n")
print(f"{'Task':<30s} {'CAMA vs Mean':>14s} {'Sum vs Mean':>14s} {'CAMA vs Sum':>14s}")
print("-" * 75)
for task, v in sum_analysis["per_task"].items():
    print(f"{task:<30s} {v.get('cama_vs_mean_d', 'n/a'):>14.4f} "
          f"{v.get('sum_vs_mean_d', 'n/a'):>14.4f} {v.get('cama_vs_sum_d', 'n/a'):>14.4f}")
print(f"\njust_use_sum verdict: {sum_analysis['just_use_sum_verdict']}")
gap_pct = sum_analysis['mean_cama_closes_gap_pct']
if math.isfinite(gap_pct):
    print(f"CAMA closes {gap_pct:.1f}% of the gap vs sum")

## Phase E: Critical Diagnostics

Identify key instabilities: Amazon epoch sensitivity, gate stasis (gates stuck at 0.5), and degenerate baselines with inflated effect sizes.

In [ ]:
def phase_e_diagnostics(all_effects: list[dict], diagnostics_data: dict) -> dict:
    """Diagnose Amazon epoch sensitivity, gate stasis, degenerate baselines."""

    # E1: Amazon epoch sensitivity (from pre-computed data)
    amazon_epoch_table = diagnostics_data.get("amazon_epoch_table", [])
    amazon_conclusion = ("10 epochs necessary: iter-4 (5 epochs) produced d=-1.38, "
                         "iter-2 and iter-5 (10 epochs) both show d>10.")

    # E2: Gate stasis (from pre-computed data)
    gate_stasis = diagnostics_data.get("gate_stasis", {
        "stasis_count": 0, "total_measured": 0, "stasis_fraction": 0.0, "affected_tasks": []
    })

    # E3: Degenerate baselines (computed from effects)
    degenerate_pairs = []
    inflated_d_count = 0
    for e in all_effects:
        if e["grade"] == "MECHANISM-ONLY":
            continue
        if e["baseline_seeds"]:
            bl_arr = e["baseline_seeds"]
            if len(bl_arr) > 1:
                bl_std = float(np.std(bl_arr, ddof=1))
                bl_mean = float(np.mean(bl_arr))
                if abs(bl_mean) > 1e-10 and bl_std / abs(bl_mean) < DEGENERATE_CV_THRESHOLD:
                    degenerate_pairs.append({
                        "exp_id": e["exp_id"], "task": e["task"],
                        "baseline_std": round(bl_std, 6), "baseline_mean": round(bl_mean, 6)
                    })
        if math.isfinite(e["d"]) and abs(e["d"]) > INFLATED_D_THRESHOLD:
            inflated_d_count += 1

    diagnostics = {
        "amazon_epoch_sensitivity": {
            "table": amazon_epoch_table,
            "conclusion": amazon_conclusion,
            "n_amazon_experiments": len(amazon_epoch_table),
        },
        "gate_stasis": gate_stasis,
        "degenerate_baselines": {
            "count": len(degenerate_pairs),
            "pairs": degenerate_pairs[:10],
            "inflated_d_count": inflated_d_count,
        },
    }
    return diagnostics

diagnostics = phase_e_diagnostics(all_effects, diagnostics_data)
print("Phase E: Diagnostics\n")
print(f"Amazon epoch sensitivity: {diagnostics['amazon_epoch_sensitivity']['n_amazon_experiments']} experiments")
for row in diagnostics['amazon_epoch_sensitivity']['table']:
    print(f"  {row['exp_id']:12s}: epochs={row['epochs']:3d}, d={row.get('d', 'n/a'):>8s}, "
          f"baseline_mae={row.get('baseline_mae', 'n/a')}, method_mae={row.get('method_mae', 'n/a')}")
gs = diagnostics['gate_stasis']
print(f"\nGate stasis: {gs['stasis_count']}/{gs['total_measured']} ({gs['stasis_fraction']:.1%})")
print(f"  Affected tasks: {gs['affected_tasks']}")
db = diagnostics['degenerate_baselines']
print(f"\nDegenerate baselines: {db['count']}, inflated |d|>{INFLATED_D_THRESHOLD}: {db['inflated_d_count']}")

## Phase F: Publication Readiness Assessment

Combine all evidence to reach a publication verdict: **publish**, **revise_and_resubmit**, **pivot**, or **abandon**.

In [ ]:
def phase_f_publication_readiness(scenarios: dict, sum_analysis: dict,
                                  diagnostics: dict, best_per_task: dict) -> dict:
    """Assess publication readiness."""
    cls_scenario = scenarios.get("scenario_3_classification", {})
    cls_d = cls_scenario.get("pooled_d", float("nan"))
    cls_p = cls_scenario.get("pooled_p_value", float("nan"))

    cls_tasks = {t: info for t, info in best_per_task.items() if info["task_type"] == "classification"}
    cls_all_positive = all(info["best_d"] > 0 for info in cls_tasks.values() if math.isfinite(info["best_d"]))

    reg_scenario = scenarios.get("scenario_4_regression", {})
    reg_d = reg_scenario.get("pooled_d", float("nan"))

    # Sum threat level
    just_use_sum = sum_analysis.get("just_use_sum_verdict", False)
    cama_vs_sum_ds = sum_analysis.get("cama_vs_sum_all_d", [])
    if just_use_sum:
        sum_threat = "high"
    elif any(d < -0.5 for d in cama_vs_sum_ds):
        sum_threat = "medium"
    else:
        sum_threat = "low"

    # Overall verdict
    green_yellow_scenario = scenarios.get("scenario_2_green_yellow", {})
    overall_d = green_yellow_scenario.get("pooled_d", float("nan"))
    overall_p = green_yellow_scenario.get("pooled_p_value", float("nan"))
    I2 = green_yellow_scenario.get("I_squared", float("nan"))

    if math.isfinite(cls_d) and cls_d > 0.4 and math.isfinite(cls_p) and cls_p < 0.05 and cls_all_positive:
        if sum_threat == "high":
            verdict = "revise_and_resubmit"
            rationale = (f"Classification-only pooled d={cls_d:.2f} (p={cls_p:.4f}) supports the narrowed claim, "
                         f"but sum threat level is {sum_threat}.")
        elif sum_threat == "medium":
            verdict = "revise_and_resubmit"
            rationale = (f"Classification pooled d={cls_d:.2f} (p={cls_p:.4f}) is promising. "
                         f"Sum threat is medium.")
        else:
            verdict = "publish"
            rationale = (f"Classification pooled d={cls_d:.2f} (p={cls_p:.4f}), all positive. "
                         f"Sum threat low.")
    elif math.isfinite(overall_d) and overall_d > 0.2:
        verdict = "revise_and_resubmit"
        rationale = (f"Overall pooled d={overall_d:.2f} (p={overall_p:.4f}), I2={I2:.1f}%. "
                     f"Effect present but heterogeneous. Regression results mixed (d={reg_d:.2f}).")
    else:
        verdict = "pivot"
        rationale = (f"Overall pooled d={overall_d:.2f} is insufficient. "
                     f"High heterogeneity (I2={I2:.1f}%).")

    claim_supported = (math.isfinite(cls_d) and cls_d > 0.4 and cls_all_positive
                       and math.isfinite(cls_p) and cls_p < 0.10)

    return {
        "claim_supported": claim_supported,
        "classification_pooled_d": round(cls_d, 4) if math.isfinite(cls_d) else None,
        "classification_pooled_p": round(cls_p, 6) if math.isfinite(cls_p) else None,
        "classification_all_positive": cls_all_positive,
        "regression_pooled_d": round(reg_d, 4) if math.isfinite(reg_d) else None,
        "sum_threat_level": sum_threat,
        "overall_verdict": verdict,
        "verdict_rationale": rationale,
        "overall_pooled_d": round(overall_d, 4) if math.isfinite(overall_d) else None,
        "overall_I_squared": round(I2, 2) if math.isfinite(I2) else None,
    }

pub_readiness = phase_f_publication_readiness(scenarios, sum_analysis, diagnostics, best_per_task)
print("Phase F: Publication Readiness\n")
print(f"  Overall verdict:            {pub_readiness['overall_verdict'].upper()}")
print(f"  Claim supported:            {pub_readiness['claim_supported']}")
print(f"  Classification pooled d:    {pub_readiness['classification_pooled_d']}")
print(f"  Classification pooled p:    {pub_readiness['classification_pooled_p']}")
print(f"  Classification all positive:{pub_readiness['classification_all_positive']}")
print(f"  Regression pooled d:        {pub_readiness['regression_pooled_d']}")
print(f"  Sum threat level:           {pub_readiness['sum_threat_level']}")
print(f"  Overall pooled d:           {pub_readiness['overall_pooled_d']}")
print(f"  Overall I-squared:          {pub_readiness['overall_I_squared']}%")
print(f"\n  Rationale: {pub_readiness['verdict_rationale']}")

## Visualization: Forest Plot and Summary

Forest plot showing per-task effect sizes with confidence intervals, plus a bar chart of pooled meta-analysis results across the 4 scenarios.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Forest plot of per-task best effect sizes ---
ax1 = axes[0]
tasks_sorted = sorted(best_per_task.keys())
y_pos = list(range(len(tasks_sorted)))
d_vals = [best_per_task[t]["best_d"] for t in tasks_sorted]
ci_lo = [best_per_task[t]["best_ci_lower"] for t in tasks_sorted]
ci_hi = [best_per_task[t]["best_ci_upper"] for t in tasks_sorted]
grades = [best_per_task[t]["grade"] for t in tasks_sorted]

grade_colors = {"GREEN": "#2ca02c", "YELLOW": "#ff7f0e", "RED": "#d62728"}
colors = [grade_colors.get(g, "#999999") for g in grades]

# Clip extreme CIs for display
clip_lo = [max(cl, -5) if math.isfinite(cl) else -5 for cl in ci_lo]
clip_hi = [min(ch, 15) if math.isfinite(ch) else 15 for ch in ci_hi]
d_clipped = [max(min(d, 15), -5) if math.isfinite(d) else 0 for d in d_vals]

for i, (d, lo, hi, c) in enumerate(zip(d_clipped, clip_lo, clip_hi, colors)):
    ax1.plot([lo, hi], [i, i], color=c, linewidth=2, solid_capstyle='round')
    ax1.plot(d, i, 'o', color=c, markersize=8, zorder=5)

ax1.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([t.replace('rel-', '') for t in tasks_sorted], fontsize=9)
ax1.set_xlabel("Cohen's d (positive = CAMA/RAMA better)")
ax1.set_title("Per-Task Best Effect Sizes (Forest Plot)")
ax1.invert_yaxis()

# Legend for grades
legend_patches = [mpatches.Patch(color=c, label=g) for g, c in grade_colors.items()]
ax1.legend(handles=legend_patches, loc='lower right', fontsize=8)

# --- Panel 2: Meta-analysis pooled effects by scenario ---
ax2 = axes[1]
sc_names = list(scenarios.keys())
sc_labels = ["GREEN only", "GREEN+YELLOW", "Classification", "Regression"]
pooled_ds = [scenarios[s]["pooled_d"] for s in sc_names]
pooled_ci_lo = [scenarios[s]["pooled_ci_lower"] for s in sc_names]
pooled_ci_hi = [scenarios[s]["pooled_ci_upper"] for s in sc_names]
i2_vals = [scenarios[s]["I_squared"] for s in sc_names]

bar_colors = ["#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e"]
bars = ax2.barh(range(len(sc_labels)), pooled_ds, color=bar_colors, alpha=0.7, height=0.5)

for i, (lo, hi) in enumerate(zip(pooled_ci_lo, pooled_ci_hi)):
    ax2.plot([lo, hi], [i, i], color='black', linewidth=1.5, solid_capstyle='round')

ax2.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.set_yticks(range(len(sc_labels)))
ax2.set_yticklabels(sc_labels, fontsize=10)
ax2.set_xlabel("Pooled Cohen's d")
ax2.set_title("Meta-Analysis: Pooled Effect Sizes")
ax2.invert_yaxis()

# Annotate with I-squared
for i, (d, i2) in enumerate(zip(pooled_ds, i2_vals)):
    x_pos = max(d, 0) + 0.1
    ax2.text(x_pos, i, f"I2={i2:.0f}%", va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig("meta_analysis_results.png", dpi=100, bbox_inches='tight')
plt.show()
print("Saved: meta_analysis_results.png")